# Interpret Thresholding

## Imports

In [3]:
%load_ext autoreload
%autoreload 2

import pandas as pd
from tqdm.auto import tqdm

from swot_toolkit.analysis import open_sites_and_dates
from swot_toolkit.pipe2 import open_output_dir


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Load Results

In [2]:
sites_dates = open_sites_and_dates("/data/swot/output")
sites_dates


{'Curua-Una': ['2024-07-13', '2025-08-14'],
 'Northeast': ['2024-05-29', '2025-07-20'],
 'Rio_Branco': ['2024-04-03', '2025-09-07'],
 'Rio_Madeira': ['2024-08-21', '2025-07-21'],
 'Rio_Negro': ['2024-11-29', '2025-08-07']}

In [15]:
results = pd.DataFrame()
for region, dates in tqdm(sites_dates.items(), desc="Regions"):
    for date in dates:
        base_dir, aoi, _ = open_output_dir(region, date)

        metrics_df = pd.read_parquet(base_dir / "results" / f"{region}_{date}_thresholds.parquet")
        metrics_df["Region"] = region
        metrics_df["Date"] = date
        results = pd.concat([results, metrics_df], axis=0)

results = results.reset_index(names=["Metric"]).set_index(["Region", "Date", "Metric"])

Regions:   0%|          | 0/5 [00:00<?, ?it/s]

In [17]:
results.head()

>0.10   >0.11   >0.12   >0.13   >0.14  \
Region    Date       Metric                                              
Curua-Una 2024-07-13 iou        0.3179  0.3247  0.3319  0.3389  0.3461   
                     f1         0.4825  0.4903  0.4984  0.5063  0.5143   
                     precision  0.3205  0.3277  0.3351  0.3425  0.3500   
                     recall     0.9752  0.9731  0.9716  0.9699  0.9690   
          2025-08-14 iou        0.4530  0.4632  0.4732  0.4822  0.4903   

                                 >0.15   >0.16   >0.17   >0.18   >0.19  ...  \
Region    Date       Metric                                             ...   
Curua-Una 2024-07-13 iou        0.3523  0.3585  0.3655  0.3711  0.3776  ...   
                     f1         0.5210  0.5278  0.5353  0.5413  0.5482  ...   
                     precision  0.3565  0.3632  0.3704  0.3766  0.3836  ...   
                     recall     0.9671  0.9656  0.9646  0.9623  0.9603  ...   
          2025-08-14 iou        0.4981  0.5057  0.5129  0.5193  0.5265  ...   

                                 >0.90   >0.91   >0.92   >0.93   >0.94  \
Region    Date       Metric                                              
Curua-Una 2024-07-13 iou        0.5501  0.5315  0.5087  0.4854  0.4547   
                     f1         0.7098  0.6941  0.6743  0.6535  0.6252   
                     precision  0.9314  0.9320  0.9322  0.9322  0.9309   
                     recall     0.5733  0.5529  0.5282  0.5031  0.4706   
          2025-08-14 iou        0.5728  0.5485  0.5254  0.4961  0.4696   

                                 >0.95   >0.96   >0.97   >0.98   >0.99  
Region    Date       Metric                                             
Curua-Una 2024-07-13 iou        0.4238  0.3851  0.3393  0.2862  0.2375  
                     f1         0.5953  0.5560  0.5067  0.4450  0.3838  
                     precision  0.9290  0.9249  0.9176  0.9079  0.8938  
                     recall     0.4380  0.3975  0.3500  0.2948  0.2444  
          2025-08-14 iou        0.4377  0.3995  0.3538  0.3035  0.2503  

[5 rows x 90 columns]

In [19]:
median_results = results.groupby(level="Metric").median()
median_results

,>0.10,>0.11,>0.12,>0.13,>0.14,>0.15,>0.16,>0.17,>0.18,>0.19,...,>0.90,>0.91,>0.92,>0.93,>0.94,>0.95,>0.96,>0.97,>0.98,>0.99
Metric,,,,,,,,,,,,,,,,,,,,,
f1,0.60240,0.60680,0.61160,0.61625,0.6207,0.62440,0.62810,0.63155,0.63480,0.63810,...,0.71910,0.7013,0.68160,0.66270,0.64320,0.61805,0.58450,0.53760,0.47325,0.39755
iou,0.43105,0.43555,0.44050,0.44535,0.4500,0.45395,0.45790,0.46150,0.46500,0.46850,...,0.56145,0.5400,0.51705,0.49555,0.47405,0.44730,0.41305,0.36780,0.31005,0.24805
precision,0.43335,0.43805,0.44325,0.44835,0.4532,0.45730,0.46145,0.46530,0.46920,0.47305,...,0.93805,0.9393,0.93990,0.94030,0.94005,0.93940,0.93715,0.93225,0.92540,0.92245
recall,0.98145,0.98035,0.97935,0.97780,0.9766,0.97560,0.97490,0.97360,0.97195,0.97040,...,0.59980,0.5784,0.55660,0.53150,0.50290,0.46910,0.42685,0.37425,0.31365,0.25060


In [28]:
median_results = median_results.rename(
    index={"iou": "IoU", "f1": "F1 Score", "precision": "Precision", "recall": "Recall"},
)

## Median graph

In [29]:
import plotly.graph_objects as go

styles = {
    "Precision": {"line": {"width": 1, "color": "blue", "dash": "dash"}},
    "Recall": {"line": {"width": 1, "color": "green", "dash": "dash"}},
    "F1 Score": {"line": {"width": 3, "color": "purple", "dash": "solid"}},
    "IoU": {"line": {"width": 3, "color": "red", "dash": "solid"}},
}

In [50]:
fig = go.Figure()

for idx, row in median_results.iterrows():
    x = [float(c[-4:]) for c in median_results.columns]
    name = idx[1] if isinstance(idx, tuple) else idx
    scatter = go.Scatter(x=x, y=row, mode="lines", name=name, line=styles[name]["line"])
    fig.add_trace(scatter)

max_f1_idx = median_results.loc["F1 Score"].idxmax()
max_f1_value = median_results.loc["F1 Score", max_f1_idx]
fig.add_annotation(
    x=float(max_f1_idx[-4:]),
    y=max_f1_value,
    text=f"Threshold={max_f1_idx[1:]}<br>F1 Score={max_f1_value:.3f}",
    showarrow=True,
    arrowhead=2,
    arrowsize=1,
    arrowwidth=2,
    arrowcolor="purple",
    bgcolor="white",
    bordercolor="purple",
    borderwidth=1,
    font={"size": 12, "color": "black"},
)

# Publication-ready layout
fig.update_layout(
    xaxis_title="Water Fraction Threshold",
    yaxis_title="Metric Value",
    plot_bgcolor="white",
    paper_bgcolor="white",
    font={"family": "Arial, sans-serif", "size": 14, "color": "black"},
    legend={
        "x": 0.02,
        "y": 0.98,
        "xanchor": "left",
        "yanchor": "top",
        "bgcolor": "rgba(255, 255, 255, 0.8)",
        "bordercolor": "black",
        "borderwidth": 1,
    },
    width=800,
    height=500,
)

# Configure axes with gridlines and precise ticks
fig.update_xaxes(
    range=[0.1, 0.9],
    showgrid=True,
    gridwidth=1,
    gridcolor="lightgray",
    showline=True,
    linewidth=2,
    linecolor="black",
    mirror=True,
    ticks="outside",
    tickwidth=2,
    tickcolor="black",
    tickmode="linear",
    tick0=0.1,
    dtick=0.1,
)

fig.update_yaxes(
    range=[0, 1.09],
    showgrid=True,
    gridwidth=1,
    gridcolor="lightgray",
    showline=True,
    linewidth=2,
    linecolor="black",
    mirror=True,
    ticks="outside",
    tickwidth=2,
    tickcolor="black",
    tickmode="linear",
    tick0=0.0,
    dtick=0.1,
)

fig

In [34]:
median_results.loc["F1 Score"].idxmax()

'>0.59'